# Method of Lines and Runge-Kutta

Connect spatial right-hand sides to Runge-Kutta time stages using NRPy
time-stepping utilities.

Navigation: [Index](../index.ipynb) |
Previous: [Wave Equation with NumPy](wave_equation_with_numpy.ipynb) |
Next: [Boundary Conditions and Convergence](boundary_conditions_and_convergence.ipynb)

## Learning Goals

- Separate space derivatives from time stepping.
- Read an RK4 table as a recipe for intermediate stages.
- See how NRPy records a time-step function for generated projects.

## Words for This Notebook

- **Method of Lines:** first approximate space derivatives, then solve the
  remaining time problem as ordinary differential equations.
- **MoL:** short for Method of Lines.
- **Stage:** one intermediate calculation inside a Runge-Kutta time step.
- **Butcher table:** the table of numbers that defines a Runge-Kutta method.
- **Right-hand side:** the formula that gives the time change of a field.
- **Function prototype:** a C function declaration that lists the name,
  return type, and arguments.

Use the code cells actively: first predict what should happen, then run the
cell, then explain the output in plain language. This predict-run-explain
pattern keeps the physics idea connected to the programming details.

## RK4 Stages and NRPy Function Storage

The right-hand side computes time derivatives. RK4 combines four stage
evaluations into one time step.

## Import Time-Stepping Tools

These imports expose the NRPy time-stepping and infrastructure writers used
below.

In [1]:
import sympy as sp
import nrpy.c_function as cfc
from nrpy.infrastructures import BHaH
from nrpy.infrastructures.BHaH.MoLtimestepping.MoL_step_forward_in_time import (
    register_CFunction_MoL_step_forward_in_time,
)
from nrpy.infrastructures.BHaH.MoLtimestepping.rk_butcher_table_dictionary import (
    generate_Butcher_tables,
    intermediate_stage_gf_names_list,
    is_diagonal_Butcher,
    validate,
)


## Inspect the RK4 Butcher Table

The table rows list stage times and stage weights. The final row lists the
weights used to combine the four stages into one RK4 update.

In [2]:
butcher_tables = generate_Butcher_tables()
rk4_table, rk4_order = butcher_tables["RK4"]
rk4_stage_rows = rk4_table[:-1]
rk4_weight_row = rk4_table[-1]
formatted_rows = []
for stage, row in enumerate(rk4_stage_rows, start=1):
    c_value = row[0]
    a_values = list(row[1:]) + [""] * (4 - len(row[1:]))
    formatted_rows.append((stage, c_value, *a_values[:4]))
print("stage c a1 a2 a3 a4")
for formatted_row in formatted_rows:
    print(*formatted_row)
print("b weights:", rk4_weight_row[1:])
print("RK4 order:", rk4_order)
print("is diagonal:", is_diagonal_Butcher(butcher_tables, "RK4"))
print(
    "intermediate storage names:",
    intermediate_stage_gf_names_list(butcher_tables, "RK4"),
)


stage c a1 a2 a3 a4
1 0    
2 1/2 1/2   
3 1/2 0 1/2  
4 1 0 0 1 
b weights: [1/6, 1/3, 1/3, 1/6]
RK4 order: 4
is diagonal: True
intermediate storage names: ['y_nplus1_running_total_gfs', 'k_odd_gfs', 'k_even_gfs']


## Validation Check

This check confirms that NRPy's RK4 entry has order 4, four stages, and a
final weights row before it is used to generate C code.

In [3]:
validate(butcher_tables, ["RK4"], verbose=False)
expected_rk4_table = [
    [0],
    [sp.Rational(1, 2), sp.Rational(1, 2)],
    [sp.Rational(1, 2), 0, sp.Rational(1, 2)],
    [1, 0, 0, 1],
    ["", sp.Rational(1, 6), sp.Rational(1, 3), sp.Rational(1, 3), sp.Rational(1, 6)],
]
print("validated RK4 order:", rk4_order)
print("stage rows:", len(rk4_stage_rows))
if rk4_order != 4:
    raise RuntimeError("Expected RK4 to report order 4.")
if rk4_table != expected_rk4_table:
    raise RuntimeError("Expected the classical RK4 Butcher coefficients.")
print("RK4 coefficient check passed")


RK4: PASSED VALIDATION
validated RK4 order: 4
stage rows: 4
RK4 coefficient check passed


## Register the RK Step-Forward Function

NRPy's stored function list holds a complete C function or workflow hook for
later file generation.

In [4]:
cfc.CFunction_dict.pop("MoL_step_forward_in_time", None)
for name in ["RK_STEP1", "RK_STEP2", "RK_STEP3", "RK_STEP4"]:
    BHaH.MoLtimestepping.rk_substep.MoL_Functions_dict.pop(name, None)
register_CFunction_MoL_step_forward_in_time(
    butcher_tables,
    "RK4",
    rhs_string="rhs_eval(commondata, params, rfmstruct, auxevol_gfs, in_gfs, rhs_gfs);",
)
registered = cfc.CFunction_dict["MoL_step_forward_in_time"]
print(
    "registered MoL function names:",
    sorted(name for name in cfc.CFunction_dict if "MoL" in name or "mol" in name),
)
print("complete C function declaration:")
print(registered.function_prototype)
print(
    "registered substep names:",
    sorted(BHaH.MoLtimestepping.rk_substep.MoL_Functions_dict),
)

registered MoL function names: ['MoL_step_forward_in_time']
complete C function declaration:
void MoL_step_forward_in_time(commondata_struct *restrict commondata, griddata_struct *restrict griddata);
registered substep names: ['RK_STEP1', 'RK_STEP2', 'RK_STEP3', 'RK_STEP4']


## Inspect the Generated RK Substep

The complete block shows how the Method of Lines update is written as a C
helper function. On a first pass, look for:

- the helper function name;
- reads from the current grid fields;
- assignments to intermediate RK storage;
- the right-hand-side function call.

In [5]:
full_function = registered.full_function
start = full_function.index("static void rk_substep_1_host")
end_marker = "} // END FUNCTION: rk_substep_1_host"
end = full_function.index(end_marker, start) + len(end_marker)
print("complete generated RK substep block:")
print(full_function[start:end])

complete generated RK substep block:
static void rk_substep_1_host(params_struct *restrict params, REAL *restrict k_odd_gfs, REAL *restrict y_n_gfs,
                              REAL *restrict y_nplus1_running_total_gfs, const REAL dt) {
  LOOP_ALL_GFS_GPS(i) {
    const REAL k_odd_gfsL = k_odd_gfs[i];
    const REAL y_n_gfsL = y_n_gfs[i];
    const REAL RK_Rational_1_6 = 1.0 / 6.0;
    const REAL RK_Rational_1_2 = 1.0 / 2.0;
    y_nplus1_running_total_gfs[i] = RK_Rational_1_6 * dt * k_odd_gfsL;
    k_odd_gfs[i] = RK_Rational_1_2 * dt * k_odd_gfsL + y_n_gfsL;
  }
} // END FUNCTION: rk_substep_1_host


The RK4 validation confirms the stage table, and the generated substep block
shows how NRPy turns those stages into C. This is the time-integration driver
used by generated wave projects.

## Learning Check

Before inspecting RK4, predict how many stages RK4 has. Then count the nonempty rows in the table.

## Continue to Boundary Conditions
- [Boundary Conditions and Convergence](boundary_conditions_and_convergence.ipynb)
- [Start-to-Finish Cartesian Wave Project](../3-wave_equation/start_to_finish_wave_cartesian.ipynb)
- [BHaH Project Anatomy](../5-infrastructures/bhah_project_anatomy.ipynb)